In [1]:
import storywrangler
from storywrangler import Storywrangler
from dotenv import load_dotenv

storywrangler.__version__

'0.0.8'

In [2]:
load_dotenv()

True

In [3]:
# Local backend (uv run uvicorn app.main:app --port 8000, from backend/).
# Drop base_url to use production. Reading data needs no API key —
# the key (from .env) is only used for registration/admin endpoints.
client = Storywrangler(base_url="http://localhost:8000")

In [4]:
client.users.whoami()

{'id': 1,
 'username': 'admin',
 'email': 'admin@storywrangler.org',
 'role': 'admin',
 'api_key': '4dd949d1-cf42-4e28-9d9a-cc9e942ef3f2',
 'is_active': True}

In [5]:
r = client.registry

In [6]:
r.list()['datasets'][0]

{'catalog': 'vcsi',
 'domain': 'babynames',
 'dataset_id': 'ngrams',
 'version': 'latest',
 'data_location': ['/users/j/s/jstonge1/babynames/metadata.ducklake.files/main/babynames/ducklake-019d25be-a8d4-7e9c-aa13-3847bd1d8fe4.parquet',
  '/users/j/s/jstonge1/babynames/metadata.ducklake.files/main/babynames/ducklake-019d25be-c181-7b2c-a00b-d32c32d3cf70.parquet'],
 'data_format': 'parquet',
 'description': 'Baby names by popularity, year, and location with entity mappings',
 'endpoint_schema': {'type': 'types-counts',
  'type_column': None,
  'count_column': None},
 'level_order': None,
 'filter_values': {'sex': ['F', 'M']},
 'created_at': '2026-04-07T16:35:14.417279+00:00',
 'updated_at': '2026-04-07T16:35:14.447812+00:00'}

In [7]:
bb = client.dataset("babynames", "ngrams")

In [8]:
bb.meta['filter_values']

{'sex': ['F', 'M']}

In [9]:
# the manifest
bb.availability

{'yearly': {'available': {'wikidata:Q176': {'min': 1980, 'max': 2024},
   'wikidata:Q30': {'min': 1880, 'max': 2024}}}}

In [10]:
# check the entity mapping
bb.adapter
# also works
# r.adapter("babynames", "ngrams")

[{'local_id': 'united_states',
  'entity_id': 'wikidata:Q30',
  'entity_name': 'United States',
  'entity_ids': ['iso:US', 'local:babynames:united_states']},
 {'local_id': 'quebec',
  'entity_id': 'wikidata:Q176',
  'entity_name': 'Quebec',
  'entity_ids': ['iso:CA-QC', 'local:babynames:quebec']}]

In [11]:
# same thing as a table
bb.adapter.df()

,local_id,entity_id,entity_name,entity_ids
0,united_states,wikidata:Q30,United States,"[iso:US, local:babynames:united_states]"
1,quebec,wikidata:Q176,Quebec,"[iso:CA-QC, local:babynames:quebec]"


In [12]:
# data endpoints mirror the API routes: /babynames/top-ngrams → .top_ngrams()
bb.top_ngrams(dates="1991", dates2="2024", sex="F", limit=5).df()

,types,counts,system
0,Ashley,43477.0,1991
1,Jessica,43395.0,1991
2,Brittany,29090.0,1991
3,Amanda,28896.0,1991
4,Samantha,25648.0,1991
5,Olivia,14718.0,2024
6,Emma,13485.0,2024
7,Amelia,12740.0,2024
8,Charlotte,12552.0,2024
9,Mia,12113.0,2024


In [13]:
# rtd compares ONE entity at two dates (for two entities, use allotax below)
bb_rtd = bb.rtd(entity="wikidata:Q30", dates="2024", dates2="1980", sex="M")

In [14]:
# every response has a .df() accessor — which names moved most since 1980?
bb_rtd.df().head(5)

,type,rank1,rank2,divergence
0,Liam,1945.0,1.0,0.000576
1,Jennifer,2.0,1078.5,-0.000475
2,Noah,511.5,2.0,0.000454
3,Oliver,905.5,3.0,0.000423
4,Michael,1.0,27.0,-0.000414


In [15]:
# allotax: rank-turbulence divergence between TWO entities (US vs Quebec boys, 2024)
result = bb.allotax(
    entity="wikidata:Q30", entity2="wikidata:Q176",
    dates="2024", dates2="2024",
    sex="M", sex2="M",
)
result["wordshift"][:3]

[{'type': 'Liam (1 ⇋ 3)', 'metric': -0.0034311517571903828},
 {'type': 'Noah (2 ⇋ 1)', 'metric': 0.0029714645859664876},
 {'type': 'Leo (35 ⇋ 2)', 'metric': 0.002885316813139781}]

In [20]:
# works for any dataset — wikimedia pageview ngrams, "hello" in the US last week
wiki = client.dataset("wikimedia", "ngrams")

In [21]:
wiki.endpoints

{'/wikimedia/top-ngrams': 'Get Top Ngrams',
 '/wikimedia/revisions': 'List Revision Articles',
 '/wikimedia/revisions/{identifier}': 'Get Revision Deltas',
 '/wikimedia/term-series': 'Term Series',
 '/wikimedia/term-series/batch': 'Term Series Batch',
 '/wikimedia/precomputed-rtd': 'Precomputed Rtd',
 '/wikimedia/semantic-timeseries': 'Semantic Timeseries',
 '/wikimedia/semantic-ngrams': 'Semantic Ngrams'}

In [22]:
wiki.term_series("hello", entity="wikidata:Q30", window=7).df()[["date", "counts", "rank"]]

,date,counts,rank
0,2026-07-05,101876,69589
1,2026-07-06,102515,71629
2,2026-07-07,101224,69939
3,2026-07-08,107206,65566
4,2026-07-09,108694,65484
5,2026-07-10,109678,64247
6,2026-07-11,110999,64248
7,2026-07-12,105695,69580
